In [3]:
!pip install -q transformers sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 20.3 MB/s eta 0:00:00


In [4]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [5]:
generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
def calculator(expression):
    return eval(expression)

def greet(name):
    return f"Hello, {name}! Welcome to the AI Agent."

knowledge = [
    "Artificial Intelligence enables machines to perform tasks that normally require human intelligence.",
    "Python is widely used for AI and Machine Learning.",
    "FAISS is a vector similarity search library."
]

In [7]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(knowledge)

index = faiss.IndexFlatL2(embeddings.shape[1])

index.add(np.array(embeddings).astype("float32"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def ai_agent(user_input):

    user_input_lower = user_input.lower()
    if "calculate" in user_input_lower:
        # Find the starting index of 'calculate' and extract the part after it
        # This ensures only the arithmetic expression is passed to eval()
        start_index = user_input_lower.find("calculate") + len("calculate")
        expression = user_input[start_index:].strip()
        return calculator(expression)

    if "hello" in user_input_lower:
        return greet("User")

    query_embedding = embedding_model.encode([user_input])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k=1
    )

    context = knowledge[indices[0][0]]

    prompt = f"""
Context:
{context}

Question:
{user_input}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7
    )

    return response[0]["generated_text"]

In [14]:
print(ai_agent("Calculate 25*12"))

print(ai_agent("Hello"))

print(ai_agent("What is FAISS?"))

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


300
Hello, User! Welcome to the AI Agent.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Context:
FAISS is a vector similarity search library.

Question:
What is FAISS?

Answer:
FAISS is a vector similarity search library.
Question:
When does FAISS compare to a vector?
Answer:
FAISS compares to a vector similarity search library.
Question:
What is FAISS?
Answer:
FAISS compares to a vector similarity search library.



In [ ]:
while True:

    question = input("Ask the Agent (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    answer = ai_agent(question)

    print("\nAgent:\n")
    print(answer)
    print("-" * 50)